# Second-Order Autoregressive Dynamics in the Complex Plane

AR(2) processes are the simplest linear recurrences with oscillatory behavior and memory:
$$
X_{t+1}=aX_t+bX_{t-1}+W_t.
$$
Stability and covariance structure are encoded by roots of the characteristic polynomial.

We simulate complex AR(2) trajectories, inspect long-run clouds, and estimate autocorrelation decay.


## Environment

Complex-valued simulation allows us to visualize trajectories as 2D curves.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

plt.rcParams["figure.dpi"] = 120
rng = np.random.default_rng(4)


## Stable parameterization

Choose radius $r<1$ and angle $\theta$, with characteristic roots $re^{\pm i\theta}$.


In [ ]:
def params_from_polar(r=0.95, theta=0.15):
    # Characteristic polynomial: z^2 - a z - b = 0
    # Roots r*exp(+/-i theta) => a=2r cos(theta), b=-r^2
    a = 2 * r * np.cos(theta)
    b = -(r**2)
    return a, b

a, b = params_from_polar(0.97, 0.12)
print("a=", a, "b=", b)


## AR(2) simulation

Generate trajectories after warm-up to approximate stationarity.


In [ ]:
def simulate_ar2(a, b, n=6000, warmup=1000, p=12, noise_scale=1.0):
    x = np.zeros((2, p), dtype=np.complex128)
    for _ in range(n + warmup):
        w = noise_scale * (rng.standard_normal(p) + 1j * rng.standard_normal(p))
        xn = a * x[-1] + b * x[-2] + w
        x = np.vstack([x, xn[None, :]])
    return x[warmup:]

X = simulate_ar2(a, b, n=9000, warmup=1500, p=12, noise_scale=0.8)


## Representative trajectories

Each particle is one realization of the same AR(2) law.


In [ ]:
fig, ax = plt.subplots(figsize=(5.8, 5.8))
for j in range(min(8, X.shape[1])):
    z = X[:, j]
    ax.plot(z.real, z.imag, lw=0.7, alpha=0.9)
ax.set_aspect("equal")
ax.set_title("Complex AR(2) sample paths")
ax.grid(alpha=0.2)
plt.show()


## Empirical autocorrelation

Estimate $\gamma(k)=\mathbb E[X_t\overline{X_{t-k}}]$ from one long realization.


In [ ]:
z = X[:, 0] - X[:, 0].mean()
K = 180
ac = np.array([np.mean(z[k:] * np.conj(z[:-k])) if k > 0 else np.mean(z * np.conj(z)) for k in range(K)])
acn = np.real(ac / ac[0])

fig, ax = plt.subplots(figsize=(6.2, 3.4))
ax.plot(acn, lw=2)
ax.set_xlabel("Lag")
ax.set_ylabel("Normalized autocorrelation")
ax.set_title("Empirical AR(2) correlation decay")
ax.grid(alpha=0.3)
plt.show()


## Interactive stability exploration

Vary $r$ and phase to observe transition from rapid damping to persistent oscillation.


In [ ]:
def preview_ar2(r=0.96, theta=0.18, n=2200):
    a, b = params_from_polar(r, theta)
    X = simulate_ar2(a, b, n=n, warmup=500, p=1, noise_scale=0.9).ravel()
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), constrained_layout=True)
    axes[0].plot(X.real[:700], lw=1.0, label="Real")
    axes[0].plot(X.imag[:700], lw=1.0, label="Imag")
    axes[0].set_title("Time series (first samples)")
    axes[0].legend()
    axes[0].grid(alpha=0.25)
    axes[1].plot(X.real, X.imag, lw=0.7)
    axes[1].set_aspect("equal")
    axes[1].set_title("Phase portrait")
    axes[1].grid(alpha=0.25)
    plt.show()

interact(
    preview_ar2,
    r=FloatSlider(min=0.75, max=0.995, step=0.005, value=0.96),
    theta=FloatSlider(min=0.02, max=0.6, step=0.01, value=0.18),
    n=IntSlider(min=1200, max=5000, step=200, value=2200),
);


## Bibliographical resources

- Peter J. Brockwell and Richard A. Davis, *Time Series: Theory and Methods*.
- Hamilton, *Time Series Analysis*.
- Percival and Walden, *Spectral Analysis for Physical Applications*.
